<a href="https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) My lane as an ML task

**Task type:** Ranking, built on a binary classifier.

The model estimates, for each page, how likely it is to be a "needs attention" page. I then sort pages by that score to make a review queue. The score matters only for the ordering, because a team can review only a limited number of pages per cycle.

**Why not clustering:** I already have a defined outcome to learn from, so this is supervised, not exploratory.

## 2) Target or proxy

**Target (proxy sketch):** `needs_review_proxy` = 1 if the page's `trend_direction` is "down", else 0.

**Why it is a proxy:** A declining trend is an observed signal, not proof that a page needs a refresh. It could be seasonal or caused by something else, so this is directional evidence only.

**Caution:** `trend_direction` is built from recent movement, so features that describe recent movement could leak the answer. I treat this as a framing sketch and will check the real label definition before modeling.

## 3) Success metric

**Primary metric:** Precision@50, meaning that of the 50 pages ranked highest, how many are true positives?

**Why this metric:** A team can review only a limited number of pages, so what matters is the quality of the top of the list, not accuracy over all pages.

**Baseline to beat:** A simple hand-written rule scored about 0.24 in the starter notebook. The model has to clearly beat this to be worth using.

**Secondary check:** Evaluate on clients not seen in training (client-holdout), so the result isn't just memorizing one client's pages.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [7]:
import os, glob
import pandas as pd

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/janaadarwish/flyrank-ml-internship.git
%cd flyrank-ml-internship

files = glob.glob("data/**/*.csv", recursive=True)
print(files)

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 169 (delta 63), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 1.86 MiB | 185.00 KiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/flyrank-ml-internship/flyrank-ml-internship
['data/raw/content_refresh_anonymized.csv']


In [8]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(list(df.columns))

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [9]:
cols = ["content_id", "client_id", "content_age_days", "days_since_last_update",
        "impressions_last_30d", "impressions_prev_30d", "ctr", "avg_position", "trend_direction"]

unit_df = df[cols].head(10)
unit_df

,content_id,client_id,content_age_days,days_since_last_update,impressions_last_30d,impressions_prev_30d,ctr,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,187,20,578,987,0.76,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,445,25,2501,5915,0.05,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,141,20,2382,6089,0.09,36.5,down
3,content_331d6c4de07b,client_19581e27de,463,22,3626,4206,0.49,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,4211,6452,0.13,44.0,down
5,content_d4084a4bc775,client_f369cb89fc,147,20,617,1009,0.03,8.5,down
6,content_9a34b442b552,client_8722616204,90,20,1,13,0.00,7.0,down
7,content_a63219c6e95a,client_19581e27de,445,22,636,632,0.06,21.2,stable
8,content_5e6c160719bc,client_6208ef0f77,90,20,5696,13828,0.09,46.0,down
9,content_c27558df2b0c,client_19581e27de,257,104,252,356,0.16,4.9,down


In [10]:
df["needs_review_proxy"] = (df["trend_direction"] == "down").astype(int)

print(df["needs_review_proxy"].value_counts())
print("Share positive:", round(df["needs_review_proxy"].mean(), 3))

needs_review_proxy
1    16262
0    13738
Name: count, dtype: int64
Share positive: 0.542


## 4) The unit of analysis

**One row = one page** (`content_id`), belonging to one client (`client_id`). Each row has measured features such as age, days since last update, impressions, CTR and average position.

The table above shows 10 real rows. The target is a sketch, `needs_review_proxy`, set to 1 when a page's trend direction is "down". About **54.2%** of pages are positive.

## 5) Why ML beats a fixed rule here

- A hand-written rule uses one or two signals and treats every client the same. The model combines many signals at once.
- In the starter run, the learned model reached Precision@50 of roughly 0.74, against roughly 0.24 for the rule. That is a directional result on the starter sample, not a guarantee.
- A rule is still the right baseline, because if the model can't beat it, the added complexity isn't justified.

**Content action:** The ranked queue tells an SEO/content team which pages to review and refresh first this cycle. A person decides whether to change each flagged page.

**Careful words:** These results are observed on an anonymized starter sample and are decision-support only. They don't show causation or predict how Google ranks pages.

## 6) Self-check

- [x] Named the task type, target/proxy and success metric
- [x] Showed the unit of analysis as a real dataframe
- [x] Explained why this is not just a fixed rule
- [x] Tied the output to a real content action